# 02. Térbeli mérések

## MIT mérünk?

Ez a fejezet a lakások **térbeli helyzetét** méri három rétegben: (1) a vasúti vágánytól mért légvonali távolságból képzett **immissziós sávok** (a zaj- és rezgésterhelés kategóriái), (2) az úthálózaton mért gyalogos **izokrónok** (hány perc séta a legközelebbi állomás/megálló: 5-10-15 perces gyűrűk), és (3) a gyalogosan elérhető szolgáltatások (**POI-k**) száma ugyanezen a 15 perces körzeten belül. Emellett az árak **térbeli eloszlását** ábrázoljuk: ponttérkép, cellánkénti medián rács és interpolált árfelület.

## MIÉRT fontos?

Egy lakás árát a belső jellemzők mellett az határozza meg, *mi érhető el tőle gyalogosan*, és *mi terheli a közvetlen környezetében*. A két hatás iránya ellentétes lehet: ugyanaz a vasúti infrastruktúra egyrészt elérhetőségi előny (gyors eljutás a városközpontba), másrészt környezeti teher (zaj, rezgés) — a kettőt ezért külön-külön, eltérő mérési logikával kell megfogni, mielőtt egy modellben együtt szerepeltetnénk őket.

## HOGYAN mérünk? Küszöbök kontra folytonos távolság

A mérés két elvet kever. A **környezeti terhelést sávosan** mérjük: a vágánytól mért légvonali távolságot rögzített küszöbökkel (0–150–300–500–1000–2000 méter) osztjuk sávokra, mert a zaj és a rezgés a forrástól távolodva erősen, nem egyenletesen csillapodik — a küszöbök a csillapodási tartományokhoz igazodnak. A **gyalogos elérhetőséget szintén küszöbösen** mérjük, de hálózaton: 1,25 m/s sétatempóval a 375 méter 5 perc, a 750 méter 10 perc, az 1125 méter 15 perc sétatáv — ezek az izokrón-sávok, mert a mindennapi döntések („elérhető-e gyalog?") valójában küszöbdöntések. Emellett mindkét dimenzióban rendelkezésre áll a **folytonos** mérték is (méterben), mert a későbbi modellek a folytonos alakot használják.

## Mi a referencia elve és a döntési szabály?

Minden sávhatást egy **referenciához** viszonyítunk: az a legtávolabbi, még kellően népes (legalább 20 megfigyelésű) sáv, amely a terheléstől gyakorlatilag függetlennek tekinthető — az izokrónoknál ez a 1125 méteren kívüli zóna. A sávok relatív helyzetét így a referencia-kontraszt adja. A fejezet minden számát a közös elemző modul térbeli méréseiből (gyorsítótárból) veszi: a notebook nem számol újra, csak megjelenít.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os; sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')
df = load_szamitott_master(); p = A.pontos_elado()


In [ ]:
import plotly.express as px
# A terület neve és fejléce dinamikusan a konfigurációból
from IPython.display import display, HTML, Markdown

area_name = get_area_metadata(os.environ.get('TDK_ACTIVE_AREA', 'kobanya'))['name']
display(HTML(render_area_header_html(area=AREA, df=df)))
print(f"Aktív terület: {area_name}  |  teljes minta: {len(df):,} sor  |  elemzési minta (pontos, eladó): {len(p):,} sor".replace(',', ' '))


## 1. Immissziós sávok és gyalogos izokrónok — a mérési táblák

**MIT látunk?** Két táblázatot. Az immissziós tábla a rögzített 0-150-300-500-1000-2000 méteres sávok elemszámait mutatja (hány lakás esik az egyes vágánytávolság-sávokba). Az izokrón-tábla hálózatonként (vasút, metró, villamos, kötöttpálya-minimum) azt, hogy hány lakásból érhető el az adott megálló 5, 10, illetve 15 perc sétával, továbbá a folytonos hálózati távolságok kitöltöttségét és mediánját.

**MIÉRT ezek a küszöbök?** Az immissziós küszöbök a zaj- és rezgéscsillapodás tartományait követik: 150 méteren belül közvetlen, 150–300 méter között erős, 300–500 méter között mérsékelt, 500–1000 és 1000–2000 méter között gyengülő, 2000 méteren túl elhanyagolható terheléssel számolunk. Az izokrón-küszöbök a sétaidőből adódnak: 75 méter/perc (1,25 m/s) mellett a 375/750/1125 méter éppen az 5/10/15 perces gyűrű.

**HOGYAN?** A sávok képzése a feldolgozó láncban, a gyorsítótárban történt; a notebook csak megjeleníti az eredményeket, oszlopdiagramokkal kiegészítve.

**Döntési szabály:** a sávhatások értelmezése mindig a referencia-kontraszt alapján történik (a legtávolabbi, legalább 20 elemű sáv, illetve a 1125 méteren kívüli zóna a viszonyítási alap); a folytonos méter-távolság a későbbi modellek bemenete.


In [ ]:
# 1. Térbeli mérések a cache-ből (A.spatial_metrics) + KPI-k
S = A.spatial_metrics()

vasut15 = int(S['izokronok'].loc[S['izokronok']['halozat'] == 'vasut', '15p_N'].iloc[0])
kp15 = int(S['izokronok'].loc[S['izokronok']['halozat'] == 'kotottpalya', '15p_N'].iloc[0])
poi15_mean = float(S['poi'].loc[S['poi']['poi'] == 'poi_15p_count', 'atlag'].iloc[0])
sav1 = int(S['immisszios_savok']['N'].iloc[0])
med_nm = float(p['nm_ar_huf'].median())
n_p = len(p)

kpi2 = [
    ("Elemzési minta", f"{n_p} db", "Pontos koordinátás eladó lakások", "#1e3a8a"),
    ("Vasút 15p séta", f"{vasut15} db", f"{vasut15/n_p*100:.1f}% lefedettség", "#ec4899"),
    ("Kötöttpálya 15p séta", f"{kp15} db", f"{kp15/n_p*100:.1f}% lefedettség", "#2563eb"),
    ("Átlagos POI 15p", f"{poi15_mean:.1f} db", "Hálózati 15-perces ellátottság", "#059669"),
    ("<150 m immisszió", f"{sav1} db", "Közvetlen vágányközeli sáv", "#dc2626"),
    ("Medián ár/m² (pontos)", fmt_huf(med_nm), "Elemzési minta", "#7c3aed"),
]
display(HTML(kpi_grid_html(kpi2)))

iso = S['izokronok'].copy()
iso.columns = ['Hálózat', '5p (N)', '10p (N)', '15p (N)', 'Folytonos kitöltött', 'Medián táv (m)']
display(HTML("<b>Gyalogos izokrónok (0-375 / 375-750 / 750-1125 m, 1,25 m/s):</b><br>"
             + iso.to_html(classes='table table-bordered table-striped', index=False)))

sav = S['immisszios_savok'].copy()
sav.columns = ['Immissziós sáv', 'N']
display(HTML("<b>Immissziós sávok (légvonali vágánytávolság, 0-150-300-500-1000-2000 m):</b><br>"
             + sav.to_html(classes='table table-bordered table-striped', index=False)))

# Oszlopdiagramok
iso_long = S['izokronok'].melt(id_vars=['halozat'], value_vars=['5p_N', '10p_N', '15p_N'],
                              var_name='izokrón', value_name='N')
iso_long['izokrón'] = iso_long['izokrón'].map({'5p_N': '0-375 m (5 perc)',
                                               '10p_N': '375-750 m (10 perc)',
                                               '15p_N': '750-1125 m (15 perc)'})
fig_iso = px.bar(iso_long, x='halozat', y='N', color='izokrón', barmode='group',
                 title='Gyalogos elérhetőség hálózatonként (lakásszám a sávokban)',
                 labels={'halozat': 'Hálózat', 'N': 'Lakások száma'}, template=PLOTLY_TEMPLATE, height=420)
fig_iso.show()

fig_sav = px.bar(sav, x='Immissziós sáv', y='N', color='N', color_continuous_scale='OrRd', text='N',
                 title='Immissziós sávok elemszámai (elemzési minta)',
                 labels={'N': 'Lakások száma'}, template=PLOTLY_TEMPLATE, height=400)
fig_sav.update_layout(coloraxis_showscale=False)
fig_sav.show()

print(f"Az elemzési minta {n_p} lakásából {vasut15} ({vasut15/n_p*100:.1f}%) éri el a legközelebbi vasútállomást "
      f"15 perc sétán belül; a közvetlen immissziós sávba (<150 m) {sav1} lakás esik.")


## 2. Szolgáltatási ellátottság (POI) a 15 perces körzetben

**MIT mérünk?** Hány darab szolgáltatás (üzlet, rendelő, gyógyszertár, iskola, óvoda, közlekedési pont stb.) érhető el gyalogosan 375, 750, illetve 1125 méteres hálózati sávon belül.

**MIÉRT?** A „15 perces város" gondolata szerint az életminőséget és a lokáció értékét az határozza meg, hogy a napi szükségletek sétatávon belül elérhetők-e. A POI-szám az elérhetőségi előny közvetlen, sávos mérőszáma.

**HOGYAN?** A POI-pontok nyílt térképi (OpenStreetMap-alapú) adatbázisból származnak; a sávonkénti darabszámok a hálózati elérhetőség alapján készültek a feldolgozó láncban, és a gyorsítótárban tárolódnak. A fejezet a sávonkénti átlagokat és a térképi eloszlást mutatja.

**Döntési szabály:** a POI-sávok kumulatívak (a 15 perces szám tartalmazza a közelebbieket is); az értelmezés az átlagos ellátottság és a területen belüli egyenlőtlenség együttes leírása.


In [ ]:
# 2. POI-statisztika a cache-ből + térkép
poi = S['poi'].copy()
poi.columns = ['POI-sáv', 'Átlagos darabszám']
poi['POI-sáv'] = poi['POI-sáv'].map({'poi_5p_count': '5p (≤375 m)',
                                     'poi_10p_count': '10p (≤750 m)',
                                     'poi_15p_count': '15p (≤1125 m)'})
display(HTML("<b>Átlagos szolgáltatásszám a hálózati sávokban (elemzési minta):</b><br>"
             + poi.to_html(classes='table table-bordered table-striped', index=False)))

fig_poi = px.bar(poi, x='POI-sáv', y='Átlagos darabszám', color='Átlagos darabszám',
                 color_continuous_scale='Greens', text='Átlagos darabszám',
                 title='Átlagos gyalogos szolgáltatás-ellátottság (db)', template=PLOTLY_TEMPLATE, height=380)
fig_poi.update_layout(coloraxis_showscale=False)
fig_poi.show()

# Térkép: 15 perces ellátottság pontonként
pmap = p.dropna(subset=['poi_15p_count', 'nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon'])
if len(pmap):
    fig_poi_map = px.scatter_map(
        pmap, lat='geokodolt_lat', lon='geokodolt_lon', color='poi_15p_count',
        size='nm_ar_huf', hover_name='cim_teljes', map_style='carto-positron',
        color_continuous_scale='Viridis',
        title='Hálózaton elérhető szolgáltatások száma 1125 m-en belül (pontonként)')
    fig_poi_map.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
    fig_poi_map.show()

at15 = float(poi.loc[poi['POI-sáv'] == '15p (≤1125 m)', 'Átlagos darabszám'].iloc[0])
at5 = float(poi.loc[poi['POI-sáv'] == '5p (≤375 m)', 'Átlagos darabszám'].iloc[0])
print(f"Az elemzési mintában átlagosan {at15:.1f} szolgáltatás érhető el 15 perc sétán belül; "
      f"a közvetlen 5 perces körzetben átlagosan {at5:.1f}.")


## 3. Ponttérkép: a fajlagos árak térbeli eloszlása

**MIT látunk?** Minden pont egy eladó lakás az elemzési mintában; a szín a négyzetméterárat, a pontméret az alapterületet jelzi.

**MIÉRT?** A nyers pontfelhő az első ellenőrzés arra, hogy az ár térben strukturált-e (gócok, gradiensek, sávok menti eltérések), mielőtt bármilyen simítást vagy modellt alkalmaznánk.

**HOGYAN?** A térkép tokenmentes webes alaptérképre épül (carto-positron); a színskála az árak 5–95. percentilisére van kalibrálva, hogy néhány szélsőséges hirdetés ne mossa el a különbségeket.

**Döntési szabály:** a ponttérkép diagnosztika; a következő lépésben a pontonkénti zajt rácsosítással és interpolációval csökkentjük.


In [ ]:
# 3. Ponttérkép (px.scatter_map, carto-positron alaptérkép, token nélkül)
def huf_fmt_2(v):
    return f"{v:,.0f} Ft/m²".replace(',', ' ') if pd.notna(v) else "—"

p_map = p.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
fig1 = px.scatter_map(
    p_map,
    lat='geokodolt_lat', lon='geokodolt_lon',
    color='nm_ar_huf', size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True,
                'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[p_map['nm_ar_huf'].quantile(0.05), p_map['nm_ar_huf'].quantile(0.95)],
    zoom=12.2, center=get_map_center(), map_style='carto-positron',
    title='Lakások térbeli elhelyezkedése és négyzetméterára (carto-positron alaptérkép)')
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

print(f"A ponttérkép {len(p_map)} lakást ábrázol; a színskála a mintabeli árak "
      f"{huf_fmt_2(p_map['nm_ar_huf'].quantile(0.05))} és {huf_fmt_2(p_map['nm_ar_huf'].quantile(0.95))} közötti tartományára kalibrált.")


## 4. Cellánkénti medián térkép és interpolált árfelület

**MIT mérünk?** Két simított ábrázolást: (a) ~150 méteres négyzetrácson a cellánkénti medián ár — csak azokat a cellákat mutatjuk, ahol legalább 3 lakás van; (b) az árak IDW-interpolációja (távolsággal fordítottan súlyozott átlag, kitevő = 2) ~75 méteres rácson, lokális síkvetületen.

**MIÉRT?** Az egyedi pontok árai zajosak; a cellamedián kiszűri az egyedi hirdetési szélsőségeket, az IDW-felület pedig folytonos árfelszínt rajzol, amelyen a magas- és alacsony árú gócok, illetve gradiensek közvetlenül leolvashatók.

**HOGYAN?** A vetítés lokális méter-koordinátákat használ (a szélességi fok koszinuszával korrigált hosszúság), így a cellák közel négyzetesek. A rácsfelbontások rögzítettek: mediánrács ~150 m, IDW-rács ~75 m; az IDW a mintapontokból számol minden rácspontra.

**Döntési szabály:** az N≥3 küszöb az egy-két pontos cellák zaját szűri; a megjelenített ár a medián (nem átlag); a színskála az 5–95. percentilisre kalibrált.


In [ ]:
# 4A. Cellánkénti medián térkép (~150 m rács, N>=3, lokális síkvetületen)
import numpy as np
import plotly.graph_objects as go

df_ar = p.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
mid_lat = float(df_ar['geokodolt_lat'].mean())
lon_scale = 111320.0 * np.cos(np.radians(mid_lat))
lat_scale = 110540.0
df_ar['x_proj'] = df_ar['geokodolt_lon'] * lon_scale
df_ar['y_proj'] = df_ar['geokodolt_lat'] * lat_scale

GRID_M = 150.0
x0 = float(df_ar['x_proj'].min()); y0 = float(df_ar['y_proj'].min())
df_ar['cell_i'] = np.floor((df_ar['x_proj'] - x0) / GRID_M).astype(int)
df_ar['cell_j'] = np.floor((df_ar['y_proj'] - y0) / GRID_M).astype(int)

cell_stats = (df_ar.groupby(['cell_i', 'cell_j'])['nm_ar_huf']
              .agg(['count', 'median'])
              .rename(columns={'count': 'N', 'median': 'median_ar'})
              .reset_index())
cell_stats = cell_stats[cell_stats['N'] >= 3].copy()
cell_stats['x_cell'] = x0 + (cell_stats['cell_i'] + 0.5) * GRID_M
cell_stats['y_cell'] = y0 + (cell_stats['cell_j'] + 0.5) * GRID_M
cell_stats['cell_lat'] = cell_stats['y_cell'] / lat_scale
cell_stats['cell_lon'] = cell_stats['x_cell'] / lon_scale

if len(cell_stats) == 0:
    print('FIGYELEM: nincs N>=3 elemszámú cella — a medián-térkép üres.')
    fig2 = go.Figure()
    fig2.add_annotation(text='Nincs elegendő adat az N>=3 rácstérképhez', showarrow=False)
    fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
else:
    _ar_lo = float(cell_stats['median_ar'].quantile(0.05))
    _ar_hi = float(cell_stats['median_ar'].quantile(0.95))
    fig2 = px.scatter_map(
        cell_stats, lat='cell_lat', lon='cell_lon',
        size='N', color='median_ar',
        hover_data={'median_ar': ':.0f', 'N': True},
        size_max=25, color_continuous_scale='Viridis', range_color=[_ar_lo, _ar_hi],
        zoom=12.2, center=get_map_center(), map_style='carto-positron',
        title='Cellánkénti medián négyzetméterár (rács ~150 m, N>=3)')
    fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
    print(f"Mediánár-rács: {len(cell_stats)} cella (N>=3, ~150 m); "
          f"cellánkénti medián tartomány: {fmt_huf(cell_stats['median_ar'].min())} – {fmt_huf(cell_stats['median_ar'].max())}")
fig2.show()

# 4B. IDW-interpolált árfelület (~75 m rács, power=2)
IDW_GRID_M = 75.0
_pad = 150.0
gx = np.arange(float(df_ar['x_proj'].min()) - _pad,
               float(df_ar['x_proj'].max()) + _pad + IDW_GRID_M / 2, IDW_GRID_M)
gy = np.arange(float(df_ar['y_proj'].min()) - _pad,
               float(df_ar['y_proj'].max()) + _pad + IDW_GRID_M / 2, IDW_GRID_M)
GX, GY = np.meshgrid(gx, gy)
grid_nodes = np.column_stack((GX.ravel(), GY.ravel()))
pts = df_ar[['x_proj', 'y_proj']].to_numpy(dtype=float)
vals = df_ar['nm_ar_huf'].to_numpy(dtype=float)

z_flat = np.full(len(grid_nodes), np.nan)
for i in range(0, len(grid_nodes), 512):
    d = grid_nodes[i:i + 512, None, :] - pts[None, :, :]
    dist2 = np.einsum('ijk,ijk->ij', d, d)
    w = 1.0 / np.maximum(dist2, 1e-9)
    z_flat[i:i + 512] = (w @ vals) / w.sum(axis=1)
Z = z_flat.reshape(GX.shape)

_z_lo, _z_hi = np.nanpercentile(Z, [2, 98])
fig2_idw = go.Figure(go.Contour(
    z=Z, x=gx, y=gy,
    colorscale='Viridis',
    contours=dict(coloring='heatmap', showlabels=False),
    zmin=_z_lo, zmax=_z_hi,
    colorbar=dict(title='Ft/m²'),
    hovertemplate='Kelet–Nyugat: %{x:.0f} m<br>Észak–Dél: %{y:.0f} m<br>Interpolált ár: %{z:,.0f} Ft/m²<extra></extra>'
))
fig2_idw.update_layout(
    title='Interpolált árfelület (IDW, power=2)',
    xaxis_title='Kelet–Nyugat (m, lokális síkvetület)',
    yaxis_title='Észak–Dél (m, lokális síkvetület)',
    height=520, template=PLOTLY_TEMPLATE)
fig2_idw.update_yaxes(scaleanchor='x', scaleratio=1.0)
fig2_idw.show()
print(f"IDW-rács: {len(gx)}x{len(gy)} = {len(grid_nodes):,} rácspont (~75 m); "
      f"interpolált tartomány (2–98%): {fmt_huf(_z_lo)} – {fmt_huf(_z_hi)}".replace(',', ' '))


## 5. Hálózati távolságok és a kerülőfaktor

**MIT mérünk?** Azt, hogy a valódi gyalogos útvonal hossza mennyivel haladja meg a légvonali távolságot. A **kerülőfaktor** a hálózati (utcahálózaton mért legrövidebb) út és a légvonali távolság hányadosa.

**MIÉRT?** A légvonali távolság szisztematikusan alulbecsüli a valós gyaloglást: az utcahálózat kerülői, a kényszerű átkelők és a zsákutcák miatt a tényleges sétaút jellemzően másfél-kétszeresére nyúlik. Ezért minden elérhetőségi elemzés hálózati távolságokra épül, és a kerülőfaktor a feldolgozó lánc állandó ellenőrző mutatója.

**HOGYAN?** A hálózati távolságok az utcahálózat-gráfon futó legrövidebb út kereséséből származnak (a feldolgozó láncban, gyorsítótárban); a notebook a medián kerülőfaktort és a hálózati távolságok mediánjait mutatja meg hálózatonként.

**Döntési szabály:** ha a kerülőfaktor szélsőséges (egynél kisebb vagy nagyon nagy), az adathibára utal; a jellemző 1,5–2 közötti érték az egészséges városi úthálózat jele.


In [ ]:
# 5. Kerülőfaktor és hálózati távolságok — számítás a cache adataiból, f-string kimenet
detour_rows = p.dropna(subset=['tavolsag_vasut_halozati_m', 'tavolsag_vasut_m'])
detour = (detour_rows['tavolsag_vasut_halozati_m'] / detour_rows['tavolsag_vasut_m'].replace(0, np.nan)).dropna()
detour_med = float(detour.median())

iso2 = S['izokronok']
net_lines = []
for _, r in iso2.iterrows():
    net_lines.append(f"{r['halozat']}: 15 perces körzetben {int(r['15p_N'])} lakás, "
                     f"folytonos távolság mediánja {r['median_tav_m']:.0f} m "
                     f"({int(r['folytonos_kitoltott'])} kitöltött érték)")
print("Hálózati távolságok hálózatonként: " + "; ".join(net_lines))
print(f"A kerülőfaktor (hálózati út / légvonali távolság) mediánja az elemzési mintán: "
      f"{detour_med:.2f} — azaz a valós sétaút mediánban közel {detour_med:.2f}-szerese "
      f"a mértani távolságnak (n = {len(detour)} megfigyelés).")


## Összefoglalás

A fejezet térbeli méréseit egyetlen mondatfolyamba rendezzük: minden szám a közös elemző modul gyorsítótárából, futásidőben illesztve.


In [ ]:
# Záró összefoglaló — minden szám a cache-ből, f-stringgel (nincs hardkódolt érték)
def _n2(v):
    return f"{int(v):,}".replace(',', ' ')

iso3 = S['izokronok']; sav3 = S['immisszios_savok']; poi3 = S['poi']
poi15 = float(poi3.loc[poi3['poi'] == 'poi_15p_count', 'atlag'].iloc[0])
vas = iso3.loc[iso3['halozat'] == 'vasut']
kp = iso3.loc[iso3['halozat'] == 'kotottpalya']
n_cells = len(cell_stats)
if n_cells:
    cell_min = fmt_huf(cell_stats['median_ar'].min())
    cell_max = fmt_huf(cell_stats['median_ar'].max())
else:
    cell_min = cell_max = "—"

print(
    f"Az elemzési minta {_n2(len(p))} lakásából {_n2(int(vas['15p_N'].iloc[0]))} lakás éri el a legközelebbi vasútállomást "
    f"15 perc sétán belül, a kötöttpályás hálózat valamely megállóját pedig {_n2(int(kp['15p_N'].iloc[0]))} lakás. "
    f"A közvetlen immissziós sávban (<150 m a vágánytól) {_n2(int(sav3['N'].iloc[0]))} lakás fekszik. "
    f"A gyalogos körzetben átlagosan {poi15:.1f} szolgáltatás érhető el 15 percen belül. "
    f"A hálózati sétaút a légvonalihoz képest mediánban {detour_med:.2f}-szeres (kerülőfaktor). "
    f"A térbeli árszerkezet {_n2(n_cells)} darab N>=3-as, ~150 méteres cellára és {_n2(len(grid_nodes))} IDW-rácspontra bontható; "
    f"a cellamediánok {cell_min} és {cell_max} között szóródnak."
)
